# 00 — Dataset Understanding

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 2 — Data Preparation & Understanding  
**Notebook goal:** Hiểu cấu trúc bộ dữ liệu Olist trước khi cleaning, feature engineering và modeling.

## Mục tiêu

Notebook này trả lời các câu hỏi:

1. Dataset có những bảng nào?
2. Mỗi bảng có bao nhiêu dòng/cột?
3. Một dòng trong mỗi bảng đại diện cho đối tượng nào (**grain**)?
4. Primary key / candidate key là gì?
5. Foreign key liên kết giữa các bảng ra sao?
6. Quan hệ 1-1, 1-N, N-1 giữa các bảng như thế nào?
7. Có missing values, exact duplicates hoặc duplicate business keys không?
8. Order lifecycle được biểu diễn bằng các timestamp nào?
9. Có bất thường logic về timestamp không?
10. Những bảng/cột nào có nguy cơ gây **data leakage**?
11. Những join nào có nguy cơ làm tăng số dòng ngoài ý muốn?
12. Những vấn đề nào cần xử lý ở bước Data Cleaning / Feature Engineering?

> **Không thực hiện trong notebook này:** imputation, xóa dữ liệu, feature engineering, train/test split, model training.

## 1. Imports & cấu hình

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Xác định project root và thư mục dữ liệu

Notebook được thiết kế để chạy khi mở Jupyter từ project root hoặc từ thư mục `notebooks/`.

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy project root có thư mục data/raw. "
        "Hãy chạy notebook bên trong repository ecommerce-delivery-delay-prediction."
    )


PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_METRICS_DIR = PROJECT_ROOT / "reports" / "metrics"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)

assert DATA_RAW_DIR.exists(), f"Không tồn tại: {DATA_RAW_DIR}"

PROJECT_ROOT: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction
DATA_RAW_DIR: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/raw


## 3. Dataset inventory

Kiểm tra các file CSV raw có trong project.

Raw data được xem là **immutable**: notebook chỉ đọc, không ghi đè lên các file trong `data/raw/`.

In [3]:
EXPECTED_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

raw_files = sorted(DATA_RAW_DIR.glob("*.csv"))

print(f"Số file CSV tìm thấy: {len(raw_files)}")
for path in raw_files:
    print("-", path.name)

Số file CSV tìm thấy: 9
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


In [4]:
missing_files = [
    filename
    for filename in EXPECTED_FILES.values()
    if not (DATA_RAW_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Thiếu các file raw sau:\n- " + "\n- ".join(missing_files)
    )

print("✓ Đã tìm thấy đầy đủ các file dataset cần thiết.")

✓ Đã tìm thấy đầy đủ các file dataset cần thiết.


## 4. Load toàn bộ dataset

Dùng tên biến theo business entity, không dùng `df1`, `df2`, ...

In [5]:
customers = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["customers"])
geolocation = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["geolocation"])
order_items = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["order_items"])
payments = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["payments"])
reviews = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["reviews"])
orders = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["orders"])
products = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["products"])
sellers = pd.read_csv(DATA_RAW_DIR / EXPECTED_FILES["sellers"])
category_translation = pd.read_csv(
    DATA_RAW_DIR / EXPECTED_FILES["category_translation"]
)

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

print("✓ Load dataset thành công.")

✓ Load dataset thành công.


## 5. Tổng quan kích thước các bảng

In [6]:
dataset_overview = (
    pd.DataFrame(
        [
            {
                "dataset": name,
                "rows": len(df),
                "columns": df.shape[1],
                "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
            }
            for name, df in datasets.items()
        ]
    )
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

display(dataset_overview)

,dataset,rows,columns,memory_mb
0,geolocation,1000163,5,50.1224
1,order_items,112650,7,18.3709
2,payments,103886,5,8.1054
3,customers,99441,5,11.0336
4,orders,99441,8,21.9468
5,reviews,99224,7,17.8424
6,products,32951,9,3.7336
7,sellers,3095,4,0.2249
8,category_translation,71,2,0.0034


### Nhận xét cần ghi nhớ

- Đây là **relational dataset**, không phải một bảng ML phẳng.
- `orders` là bảng trung tâm.
- `order_items`, `payments`, `reviews` có thể có nhiều dòng cho một `order_id`.
- `geolocation` có grain riêng và cần kiểm tra kỹ trước khi join.
- Final ML dataset sau này phải được xây về **order-level** nếu mục tiêu là dự đoán giao trễ cho từng order.

## 6. Helper functions cho Dataset Understanding

In [7]:
def inspect_dataframe(df: pd.DataFrame, name: str, sample_size: int = 5) -> None:
    print(f"DATASET: {name}")
    print("=" * 100)
    print(f"Shape: {df.shape}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    display(df.dtypes.rename("dtype").to_frame())

    print("\nSample:")
    display(df.head(sample_size))


def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    missing_count = df.isna().sum()
    report = pd.DataFrame(
        {
            "missing_count": missing_count,
            "missing_pct": missing_count / len(df) * 100,
        }
    )

    return (
        report[report["missing_count"] > 0]
        .sort_values(["missing_pct", "missing_count"], ascending=False)
    )


def duplicate_report(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "dataset": name,
                "rows": len(df),
                "exact_duplicate_rows": int(df.duplicated().sum()),
            }
            for name, df in datasets.items()
        ]
    ).sort_values("exact_duplicate_rows", ascending=False)


def key_check(
    df: pd.DataFrame,
    columns: str | list[str],
    table_name: str,
) -> dict:
    subset = [columns] if isinstance(columns, str) else columns

    return {
        "table": table_name,
        "candidate_key": " + ".join(subset),
        "rows": len(df),
        "unique_keys": len(df[subset].drop_duplicates()),
        "duplicate_key_rows": int(df.duplicated(subset=subset).sum()),
        "is_unique": not df.duplicated(subset=subset).any(),
    }


def foreign_key_check(
    child_df: pd.DataFrame,
    child_col: str,
    parent_df: pd.DataFrame,
    parent_col: str,
    relation_name: str,
) -> dict:
    child_values = child_df[child_col].dropna()
    parent_values = parent_df[parent_col].dropna()

    unknown_mask = ~child_values.isin(parent_values)

    return {
        "relationship": relation_name,
        "child_rows_non_null": len(child_values),
        "unknown_fk_rows": int(unknown_mask.sum()),
        "unknown_fk_unique_values": int(child_values[unknown_mask].nunique()),
        "integrity_ok": bool((~unknown_mask).all()),
    }

# 7. Inspect schema của từng bảng

Mục tiêu: nhìn schema, data types và một vài dòng mẫu trước khi đưa ra bất kỳ giả định nào về grain hoặc keys.

In [8]:
for name, df in datasets.items():
    inspect_dataframe(df, name)
    print("\n")

DATASET: customers
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str



Sample:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP




DATASET: geolocation
Shape: (1000163, 5)

Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP




DATASET: category_translation
Shape: (71, 2)

Columns:
['product_category_name', 'product_category_name_english']

Data types:


,dtype
product_category_name,str
product_category_name_english,str



Sample:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


# 8. Grain và candidate keys

**Grain** = một row trong bảng đại diện cho cái gì.

Ta không chỉ dựa vào mô tả dataset; cần kiểm tra candidate key bằng dữ liệu thực tế.

In [9]:
key_checks = pd.DataFrame(
    [
        key_check(orders, "order_id", "orders"),
        key_check(customers, "customer_id", "customers"),
        key_check(products, "product_id", "products"),
        key_check(sellers, "seller_id", "sellers"),
        key_check(order_items, ["order_id", "order_item_id"], "order_items"),
        key_check(payments, ["order_id", "payment_sequential"], "payments"),
        key_check(reviews, "review_id", "reviews"),
        key_check(
            category_translation,
            "product_category_name",
            "category_translation",
        ),
        key_check(
            geolocation,
            "geolocation_zip_code_prefix",
            "geolocation",
        ),
    ]
)

display(key_checks)

,table,candidate_key,rows,unique_keys,duplicate_key_rows,is_unique
0,orders,order_id,99441,99441,0,True
1,customers,customer_id,99441,99441,0,True
2,products,product_id,32951,32951,0,True
3,sellers,seller_id,3095,3095,0,True
4,order_items,order_id + order_item_id,112650,112650,0,True
5,payments,order_id + payment_sequential,103886,103886,0,True
6,reviews,review_id,99224,98410,814,False
7,category_translation,product_category_name,71,71,0,True
8,geolocation,geolocation_zip_code_prefix,1000163,19015,981148,False


### Grain dự kiến cần xác minh từ kết quả

| Table | Grain dự kiến |
|---|---|
| `orders` | 1 row = 1 order |
| `customers` | 1 row = 1 `customer_id` record |
| `products` | 1 row = 1 product |
| `sellers` | 1 row = 1 seller |
| `order_items` | 1 row = 1 item position trong một order |
| `payments` | 1 row = 1 payment record / sequence |
| `reviews` | 1 row = 1 review record |
| `geolocation` | nhiều observation có thể dùng chung zip-code prefix |
| `category_translation` | 1 row = 1 category mapping |

> Không được giả định `geolocation_zip_code_prefix` là unique nếu kết quả kiểm tra cho thấy ngược lại.

# 9. Quan hệ customer identity

Olist có cả `customer_id` và `customer_unique_id`. Hai cột này không nhất thiết đại diện cùng một khái niệm.

Kiểm tra số lượng unique để hiểu identity.

In [10]:
customer_identity_summary = pd.Series(
    {
        "customer_rows": len(customers),
        "customer_id_unique": customers["customer_id"].nunique(),
        "customer_unique_id_unique": customers["customer_unique_id"].nunique(),
        "customer_id_is_unique": customers["customer_id"].is_unique,
    },
    name="value",
)

display(customer_identity_summary.to_frame())

,value
customer_rows,99441
customer_id_unique,99441
customer_unique_id_unique,96096
customer_id_is_unique,True


In [11]:
orders_per_unique_customer = (
    customers.groupby("customer_unique_id")
    .size()
    .rename("customer_records")
)

display(orders_per_unique_customer.describe())
display(
    orders_per_unique_customer
    .sort_values(ascending=False)
    .head(10)
    .to_frame()
)

count   96,096.0000
mean         1.0348
std          0.2144
min          1.0000
25%          1.0000
50%          1.0000
75%          1.0000
max         17.0000
Name: customer_records, dtype: float64

,customer_records
customer_unique_id,
8d50f5eadf50201ccdcedfb9e2ac8455,17
3e43e6105506432c953e165fb2acf44c,9
6469f99c1f9dfae7733b25662e7f1782,7
1b6c7548a2a1f9037c1fd3ddfed95f33,7
ca77025e7201e3b30c44b472ff346268,7
dc813062e0fc23409cd255f7f53c7074,6
f0e310a6839dce9de1638e0fe5ab282a,6
63cfc61cee11cbe306bff5857d00bfe4,6
de34b16117594161a6a89c50b289d35a,6


### Modeling implication

Nếu `customer_unique_id` xuất hiện nhiều lần, đây là tín hiệu rằng một khách hàng thực có thể có nhiều `customer_id` record.

Điều này quan trọng nếu Phase 3 muốn xây **historical customer features**. Khi đó phải đảm bảo chỉ sử dụng lịch sử có trước prediction point để tránh leakage.

# 10. Foreign-key integrity

Kiểm tra các foreign key quan trọng có tồn tại trong parent table tương ứng hay không.

In [12]:
fk_checks = pd.DataFrame(
    [
        foreign_key_check(
            orders,
            "customer_id",
            customers,
            "customer_id",
            "orders.customer_id → customers.customer_id",
        ),
        foreign_key_check(
            order_items,
            "order_id",
            orders,
            "order_id",
            "order_items.order_id → orders.order_id",
        ),
        foreign_key_check(
            order_items,
            "product_id",
            products,
            "product_id",
            "order_items.product_id → products.product_id",
        ),
        foreign_key_check(
            order_items,
            "seller_id",
            sellers,
            "seller_id",
            "order_items.seller_id → sellers.seller_id",
        ),
        foreign_key_check(
            payments,
            "order_id",
            orders,
            "order_id",
            "payments.order_id → orders.order_id",
        ),
        foreign_key_check(
            reviews,
            "order_id",
            orders,
            "order_id",
            "reviews.order_id → orders.order_id",
        ),
    ]
)

display(fk_checks)

,relationship,child_rows_non_null,unknown_fk_rows,unknown_fk_unique_values,integrity_ok
0,orders.customer_id → customers.customer_id,99441,0,0,True
1,order_items.order_id → orders.order_id,112650,0,0,True
2,order_items.product_id → products.product_id,112650,0,0,True
3,order_items.seller_id → sellers.seller_id,112650,0,0,True
4,payments.order_id → orders.order_id,103886,0,0,True
5,reviews.order_id → orders.order_id,99224,0,0,True


# 11. Cardinality

Kiểm tra số bản ghi con trên mỗi order để hiểu các quan hệ 1-N.

In [13]:
items_per_order = order_items.groupby("order_id").size().rename("items_per_order")
payments_per_order = payments.groupby("order_id").size().rename("payments_per_order")
reviews_per_order = reviews.groupby("order_id").size().rename("reviews_per_order")

cardinality_summary = pd.DataFrame(
    {
        "order_items": items_per_order.describe(),
        "payments": payments_per_order.describe(),
        "reviews": reviews_per_order.describe(),
    }
)

display(cardinality_summary)

,order_items,payments,reviews
count,"98,666.0000","99,440.0000","98,673.0000"
mean,1.1417,1.0447,1.0056
std,0.5385,0.3812,0.0751
min,1.0000,1.0000,1.0000
25%,1.0000,1.0000,1.0000
50%,1.0000,1.0000,1.0000
75%,1.0000,1.0000,1.0000
max,21.0000,29.0000,3.0000


In [14]:
print("Top orders theo số item:")
display(items_per_order.sort_values(ascending=False).head(10).to_frame())

print("\nTop orders theo số payment records:")
display(payments_per_order.sort_values(ascending=False).head(10).to_frame())

print("\nTop orders theo số review records:")
display(reviews_per_order.sort_values(ascending=False).head(10).to_frame())

Top orders theo số item:


,items_per_order
order_id,
8272b63d03f5f79c56e9e4120aec44ef,21
1b15974a0141d54e36626dca3fdc731a,20
ab14fdcfbe524636d65ee38360e22ce8,20
9ef13efd6949e4573a18964dd1bbe7f5,15
428a2f660dc84138d969ccd69a0ab6d5,15
73c8ab38f07dc94389065f7eba4f297a,14
9bdc4d4c71aa1de4606060929dee888c,14
37ee401157a3a0b28c9c6d0ed8c3b24b,13
637617b3ffe9e2f7a2411243829226d0,12



Top orders theo số payment records:


,payments_per_order
order_id,
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
285c2e15bebd4ac83635ccc563dc71f4,22
895ab968e7bb0d5659d16cd74cd1650c,21
fedcd9f7ccdc8cba3a18defedd1a5547,19
ee9ca989fc93ba09a6eddc250ce01742,19
4bfcba9e084f46c8e3cb49b0fa6e6159,15
21577126c19bf11a0b91592e5844ba78,15
4689b1816de42507a7d63a4617383c59,14



Top orders theo số review records:


,reviews_per_order
order_id,
03c939fd7fd3b38f8485a0f95798f1f6,3
c88b1d1b157a9999ce368f218a407141,3
8e17072ec97ce29f0e1f111e598b0c85,3
df56136b8031ecd28e200bb18e6ddb2e,3
075a544c5f4ed4bb75f82b160465fe76,2
823384842bb58973b0c88e31c6f94da9,2
d61b915b69851aec8a8865f36cfd793e,2
8e350e1e4254bd7c68913b98bde7d3a7,2
7f13a20e25350f4a55fb2a7c9a2e8d88,2


## Join risk: row multiplication

Nếu một order có:

- 2 `order_items`
- 3 `payments`

và ta join trực tiếp cả hai bảng vào `orders`, order đó có thể biến thành **2 × 3 = 6 rows**.

Vì vậy ở Phase 3, các bảng one-to-many như `order_items`, `payments` nên được **aggregate độc lập về order-level trước khi merge** vào modeling table.

# 12. Missing values

Missing value không đồng nghĩa data error. Cần hiểu missing trong business context trước khi quyết định xử lý.

In [15]:
missing_summaries = {}

for name, df in datasets.items():
    report = missing_report(df)
    missing_summaries[name] = report

    print(f"\n{name}")
    print("-" * 80)

    if report.empty:
        print("Không có missing values.")
    else:
        display(report)


customers
--------------------------------------------------------------------------------
Không có missing values.

geolocation
--------------------------------------------------------------------------------
Không có missing values.

order_items
--------------------------------------------------------------------------------
Không có missing values.

payments
--------------------------------------------------------------------------------
Không có missing values.

reviews
--------------------------------------------------------------------------------


,missing_count,missing_pct
review_comment_title,87656,88.3415
review_comment_message,58247,58.7025



orders
--------------------------------------------------------------------------------


,missing_count,missing_pct
order_delivered_customer_date,2965,2.9817
order_delivered_carrier_date,1783,1.7930
order_approved_at,160,0.1609



products
--------------------------------------------------------------------------------


,missing_count,missing_pct
product_category_name,610,1.8512
product_name_lenght,610,1.8512
product_description_lenght,610,1.8512
product_photos_qty,610,1.8512
product_weight_g,2,0.0061
product_length_cm,2,0.0061
product_height_cm,2,0.0061
product_width_cm,2,0.0061



sellers
--------------------------------------------------------------------------------
Không có missing values.

category_translation
--------------------------------------------------------------------------------
Không có missing values.


### Cần chú ý đặc biệt

- Missing ở delivery timestamp có thể liên quan đến order chưa được giao / bị cancel / unavailable.
- Missing ở product attributes cần điều tra trước khi imputation.
- Missing review text có thể đơn giản là khách hàng không viết comment.
- Không dùng `fillna(0)` một cách mặc định.

# 13. Exact duplicates

Phân biệt:

- **Exact duplicate row**: toàn bộ row bị lặp.
- **Duplicate business key**: key xuất hiện nhiều lần.

Duplicate business key có thể hoàn toàn hợp lệ nếu grain của bảng là one-to-many.

In [16]:
display(duplicate_report(datasets))

,dataset,rows,exact_duplicate_rows
1,geolocation,1000163,261831
0,customers,99441,0
2,order_items,112650,0
3,payments,103886,0
4,reviews,99224,0
5,orders,99441,0
6,products,32951,0
7,sellers,3095,0
8,category_translation,71,0


# 14. Orders — bảng trung tâm

Phân tích kỹ `orders` vì target và prediction point đều phụ thuộc vào bảng này.

In [17]:
display(orders.head())
display(orders.dtypes.rename("dtype").to_frame())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,str
order_approved_at,str
order_delivered_carrier_date,str
order_delivered_customer_date,str
order_estimated_delivery_date,str


## 14.1 Order status

In [18]:
order_status_counts = orders["order_status"].value_counts(dropna=False)
order_status_pct = orders["order_status"].value_counts(
    normalize=True,
    dropna=False,
).mul(100)

order_status_summary = pd.concat(
    [
        order_status_counts.rename("count"),
        order_status_pct.rename("pct"),
    ],
    axis=1,
)

display(order_status_summary)

,count,pct
order_status,,
delivered,96478,97.0203
shipped,1107,1.1132
canceled,625,0.6285
unavailable,609,0.6124
invoiced,314,0.3158
processing,301,0.3027
created,5,0.0050
approved,2,0.0020


### Target implication

Target `late_delivery` yêu cầu biết **actual customer delivery time** và **estimated delivery time**.

Do đó các order không có delivery outcome hoàn chỉnh không thể tự động bị gán vào class `0`. Population đủ điều kiện modeling phải được xác định ở bước Target Investigation / Cleaning.

# 15. Parse timestamp trong memory

Không thay đổi file raw. Chỉ tạo copy trong memory để kiểm tra lifecycle.

In [19]:
orders_dt = orders.copy()

timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in timestamp_columns:
    orders_dt[col] = pd.to_datetime(
        orders_dt[col],
        errors="coerce",
    )

display(orders_dt[timestamp_columns].dtypes.rename("dtype").to_frame())

,dtype
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


## 15.1 Date ranges

In [20]:
date_ranges = pd.DataFrame(
    {
        col: {
            "min": orders_dt[col].min(),
            "max": orders_dt[col].max(),
            "missing": orders_dt[col].isna().sum(),
        }
        for col in timestamp_columns
    }
).T

display(date_ranges)

,min,max,missing
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


### Modeling implication

Order data có thứ tự thời gian rõ ràng. Vì production sẽ dùng lịch sử để dự đoán các order tương lai, Phase 3/4 nên ưu tiên **chronological split / time-based validation** thay vì random split thuần túy.

# 16. Order lifecycle

Lifecycle logic dự kiến:

`purchase → approval → carrier handoff → customer delivery`

`estimated delivery` là deadline được hứa với khách hàng, không phải actual event.

In [21]:
lifecycle_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

display(
    orders_dt[lifecycle_columns]
    .sample(min(10, len(orders_dt)), random_state=42)
)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
52263,b9a6c5f5df52c7226ac85aee7524c27f,delivered,2018-06-12 20:07:44,2018-06-12 20:44:26,2018-06-13 13:09:00,2018-06-19 12:44:08,2018-07-17
46645,261e71d2349c713eafa9f3df5972b95d,delivered,2018-01-20 12:15:57,2018-01-20 12:37:13,2018-01-25 21:42:52,2018-01-30 11:32:35,2018-02-15
37546,67b50899f52995848c427e361e10dde3,delivered,2018-06-16 21:24:10,2018-06-16 21:36:59,2018-06-21 13:55:00,2018-06-27 13:17:27,2018-07-16
94756,32733fc014b67ef70fa6039dd8c6ba82,delivered,2017-08-30 21:12:28,2017-08-31 02:50:24,2017-09-12 20:16:46,2017-09-25 17:53:23,2017-09-22
14771,39a70e9e9b729b11dee34ac12478597f,delivered,2017-08-10 21:26:25,2017-08-10 21:44:13,2017-08-11 19:11:16,2017-08-22 16:45:00,2017-09-12
36263,80000ae9d118d79953522e35cce34f13,delivered,2017-02-27 11:33:35,2017-02-27 11:45:10,2017-03-01 11:03:51,2017-03-08 08:51:57,2017-03-20
98556,2bfd14409ba8ba1153ce42b2abc44bb8,delivered,2017-12-18 17:43:43,2017-12-18 18:52:09,2017-12-19 19:04:27,2018-01-05 13:46:27,2018-01-18
23747,faa01b7a24d0ba51d0d0ec652cd9b745,delivered,2018-07-10 18:14:47,2018-07-12 03:30:17,2018-07-12 07:30:00,2018-07-25 11:52:20,2018-08-06
50315,06aa0aa8c22027ebb953d501c4238512,delivered,2018-04-25 19:43:19,2018-04-25 20:31:06,2018-04-26 14:39:00,2018-04-27 17:37:28,2018-05-11
6501,3762c0ef92b5bc397671e6d7e6d5672f,delivered,2017-08-15 21:32:14,2017-08-15 23:30:19,2017-08-17 18:47:31,2017-08-22 22:42:13,2017-09-12


# 17. Timestamp consistency checks

Các check dưới đây **không tự động xóa row**. Chúng chỉ phát hiện records cần điều tra.

In [22]:
timestamp_checks = {
    "approval_before_purchase": (
        orders_dt["order_approved_at"]
        < orders_dt["order_purchase_timestamp"]
    ),
    "carrier_before_approval": (
        orders_dt["order_delivered_carrier_date"]
        < orders_dt["order_approved_at"]
    ),
    "customer_delivery_before_carrier": (
        orders_dt["order_delivered_customer_date"]
        < orders_dt["order_delivered_carrier_date"]
    ),
    "customer_delivery_before_purchase": (
        orders_dt["order_delivered_customer_date"]
        < orders_dt["order_purchase_timestamp"]
    ),
    "estimated_delivery_before_purchase": (
        orders_dt["order_estimated_delivery_date"]
        < orders_dt["order_purchase_timestamp"]
    ),
}

timestamp_quality_summary = pd.DataFrame(
    [
        {
            "check": name,
            "affected_rows": int(mask.fillna(False).sum()),
            "affected_pct": float(mask.fillna(False).mean() * 100),
        }
        for name, mask in timestamp_checks.items()
    ]
)

display(timestamp_quality_summary)

,check,affected_rows,affected_pct
0,approval_before_purchase,0,0.0000
1,carrier_before_approval,1359,1.3666
2,customer_delivery_before_carrier,23,0.0231
3,customer_delivery_before_purchase,0,0.0000
4,estimated_delivery_before_purchase,0,0.0000


In [23]:
for name, mask in timestamp_checks.items():
    affected = orders_dt.loc[mask.fillna(False), lifecycle_columns]

    if not affected.empty:
        print(f"\n{name}: {len(affected)} rows")
        display(affected.head(10))


carrier_before_approval: 1359 rows


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19



customer_delivery_before_carrier: 23 rows


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


# 18. Delivery timestamp availability theo order status

Điều tra missing delivery timestamp trong business context.

In [24]:
delivery_availability_by_status = (
    orders_dt.assign(
        has_customer_delivery=orders_dt[
            "order_delivered_customer_date"
        ].notna(),
        has_estimated_delivery=orders_dt[
            "order_estimated_delivery_date"
        ].notna(),
        has_approval=orders_dt["order_approved_at"].notna(),
    )
    .groupby("order_status", dropna=False)
    .agg(
        orders=("order_id", "size"),
        with_customer_delivery=("has_customer_delivery", "sum"),
        with_estimated_delivery=("has_estimated_delivery", "sum"),
        with_approval=("has_approval", "sum"),
    )
)

delivery_availability_by_status["customer_delivery_pct"] = (
    delivery_availability_by_status["with_customer_delivery"]
    / delivery_availability_by_status["orders"]
    * 100
)

delivery_availability_by_status["estimated_delivery_pct"] = (
    delivery_availability_by_status["with_estimated_delivery"]
    / delivery_availability_by_status["orders"]
    * 100
)

display(delivery_availability_by_status)

,orders,with_customer_delivery,with_estimated_delivery,with_approval,customer_delivery_pct,estimated_delivery_pct
order_status,,,,,,
approved,2,0,2,2,0.0000,100.0000
canceled,625,6,625,484,0.9600,100.0000
created,5,0,5,0,0.0000,100.0000
delivered,96478,96470,96478,96464,99.9917,100.0000
invoiced,314,0,314,314,0.0000,100.0000
processing,301,0,301,301,0.0000,100.0000
shipped,1107,0,1107,1107,0.0000,100.0000
unavailable,609,0,609,609,0.0000,100.0000


# 19. Target feasibility check — chưa tạo target chính thức

Project định nghĩa:

`late_delivery = actual_customer_delivery > estimated_delivery`

Nhưng trước khi tạo modeling target chính thức, cần kiểm tra population nào có đủ hai timestamp.

Cell dưới đây chỉ đánh giá **target availability**, chưa thay đổi dataset.

In [25]:
target_available_mask = (
    orders_dt["order_delivered_customer_date"].notna()
    & orders_dt["order_estimated_delivery_date"].notna()
)

target_feasibility = pd.Series(
    {
        "total_orders": len(orders_dt),
        "orders_with_target_available": int(target_available_mask.sum()),
        "orders_without_target_available": int((~target_available_mask).sum()),
        "target_available_pct": float(target_available_mask.mean() * 100),
    },
    name="value",
)

display(target_feasibility.to_frame())

,value
total_orders,"99,441.0000"
orders_with_target_available,"96,476.0000"
orders_without_target_available,"2,965.0000"
target_available_pct,97.0183


In [26]:
target_feasibility_by_status = (
    orders_dt.assign(target_available=target_available_mask)
    .groupby("order_status", dropna=False)
    .agg(
        orders=("order_id", "size"),
        target_available=("target_available", "sum"),
    )
)

target_feasibility_by_status["target_available_pct"] = (
    target_feasibility_by_status["target_available"]
    / target_feasibility_by_status["orders"]
    * 100
)

display(target_feasibility_by_status)

,orders,target_available,target_available_pct
order_status,,,
approved,2,0,0.0000
canceled,625,6,0.9600
created,5,0,0.0000
delivered,96478,96470,99.9917
invoiced,314,0,0.0000
processing,301,0,0.0000
shipped,1107,0,0.0000
unavailable,609,0,0.0000


# 20. Geolocation — kiểm tra grain và duplication

Không được join raw geolocation trực tiếp vào customer/seller trước khi hiểu cardinality.

In [27]:
geo_zip_stats = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .size()
    .rename("rows_per_zip_prefix")
)

display(geo_zip_stats.describe())
display(
    geo_zip_stats
    .sort_values(ascending=False)
    .head(10)
    .to_frame()
)

count   19,015.0000
mean        52.5986
std         72.0579
min          1.0000
25%         10.0000
50%         29.0000
75%         66.5000
max      1,146.0000
Name: rows_per_zip_prefix, dtype: float64

,rows_per_zip_prefix
geolocation_zip_code_prefix,
24220,1146
24230,1102
38400,965
35500,907
11680,879
22631,832
30140,810
11740,788
38408,773


In [28]:
geo_coordinate_variation = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(
        rows=("geolocation_zip_code_prefix", "size"),
        unique_lat=("geolocation_lat", "nunique"),
        unique_lng=("geolocation_lng", "nunique"),
        unique_city=("geolocation_city", "nunique"),
        unique_state=("geolocation_state", "nunique"),
    )
    .sort_values("rows", ascending=False)
)

display(geo_coordinate_variation.head(20))

,rows,unique_lat,unique_lng,unique_city,unique_state
geolocation_zip_code_prefix,,,,,
24220,1146,410,411,2,1
24230,1102,327,327,2,1
38400,965,746,745,2,1
35500,907,726,726,2,1
11680,879,726,726,1,1
22631,832,141,142,1,1
30140,810,379,380,1,1
11740,788,666,666,2,1
38408,773,600,599,2,1


### Geolocation implication

Nếu một zip prefix có nhiều dòng hoặc nhiều tọa độ, Phase 3 cần tạo một bảng lookup duy nhất theo zip prefix, ví dụ bằng median latitude/longitude hoặc một quy tắc aggregation có lý do rõ ràng.

Không join raw `geolocation` trực tiếp với `customers`/`sellers` vì có thể làm duplicate order rows.

# 21. Product attributes

Kiểm tra các numeric product attributes để phát hiện missing và giá trị không hợp lệ về mặt logic.

In [29]:
product_numeric_columns = [
    col
    for col in [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ]
    if col in products.columns
]

display(products[product_numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
product_name_lenght,"32,341.0000",48.4769,10.2457,5.0000,42.0000,51.0000,57.0000,76.0000
product_description_lenght,"32,341.0000",771.4953,635.1152,4.0000,339.0000,595.0000,972.0000,"3,992.0000"
product_photos_qty,"32,341.0000",2.1890,1.7368,1.0000,1.0000,1.0000,3.0000,20.0000
product_weight_g,"32,949.0000","2,276.4725","4,282.0387",0.0000,300.0000,700.0000,"1,900.0000","40,425.0000"
product_length_cm,"32,949.0000",30.8151,16.9145,7.0000,18.0000,25.0000,38.0000,105.0000
product_height_cm,"32,949.0000",16.9377,13.6376,2.0000,8.0000,13.0000,21.0000,105.0000
product_width_cm,"32,949.0000",23.1967,12.0790,6.0000,15.0000,20.0000,30.0000,118.0000


In [30]:
product_invalid_summary = []

for col in product_numeric_columns:
    series = products[col]

    product_invalid_summary.append(
        {
            "column": col,
            "missing": int(series.isna().sum()),
            "zero_values": int((series == 0).sum()),
            "negative_values": int((series < 0).sum()),
        }
    )

display(pd.DataFrame(product_invalid_summary))

,column,missing,zero_values,negative_values
0,product_name_lenght,610,0,0
1,product_description_lenght,610,0,0
2,product_photos_qty,610,0,0
3,product_weight_g,2,4,0
4,product_length_cm,2,0,0
5,product_height_cm,2,0,0
6,product_width_cm,2,0,0


# 22. Order item numeric sanity checks

In [31]:
order_item_numeric_cols = [
    col
    for col in ["price", "freight_value"]
    if col in order_items.columns
]

display(order_items[order_item_numeric_cols].describe().T)

order_item_numeric_checks = pd.DataFrame(
    [
        {
            "column": col,
            "missing": int(order_items[col].isna().sum()),
            "zero_values": int((order_items[col] == 0).sum()),
            "negative_values": int((order_items[col] < 0).sum()),
        }
        for col in order_item_numeric_cols
    ]
)

display(order_item_numeric_checks)

,count,mean,std,min,25%,50%,75%,max
price,"112,650.0000",120.6537,183.6339,0.8500,39.9000,74.9900,134.9000,"6,735.0000"
freight_value,"112,650.0000",19.9903,15.8064,0.0000,13.0800,16.2600,21.1500,409.6800


,column,missing,zero_values,negative_values
0,price,0,0,0
1,freight_value,0,383,0


# 23. Payment structure

In [32]:
display(payments["payment_type"].value_counts(dropna=False).to_frame("count"))

if "payment_installments" in payments.columns:
    display(payments["payment_installments"].describe())

if "payment_value" in payments.columns:
    display(payments["payment_value"].describe())

,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


count   103,886.0000
mean          2.8533
std           2.6871
min           0.0000
25%           1.0000
50%           1.0000
75%           4.0000
max          24.0000
Name: payment_installments, dtype: float64

count   103,886.0000
mean        154.1004
std         217.4941
min           0.0000
25%          56.7900
50%         100.0000
75%         171.8375
max      13,664.0800
Name: payment_value, dtype: float64

# 24. Review table và leakage

Review information được tạo sau order processing/delivery, trong khi prediction point của project là `order_approved_at`.

Vì vậy review columns không được dùng làm model input.

In [33]:
display(reviews.head())

review_time_columns = [
    col
    for col in ["review_creation_date", "review_answer_timestamp"]
    if col in reviews.columns
]

print("Review columns:")
print(reviews.columns.tolist())

print("\nReview timestamp columns:")
print(review_time_columns)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


Review columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Review timestamp columns:
['review_creation_date', 'review_answer_timestamp']


# 25. Feature availability & leakage registry

Registry này ghi lại các cột rõ ràng có vai trò đặc biệt.

Danh sách feature cuối cùng sẽ được chốt ở Phase 3, sau khi xây order-level feature table.

In [34]:
feature_availability_registry = pd.DataFrame(
    [
        {
            "column": "order_purchase_timestamp",
            "role": "candidate_feature",
            "available_at_prediction": True,
            "reason": "Có trước order approval.",
        },
        {
            "column": "order_approved_at",
            "role": "prediction_point",
            "available_at_prediction": True,
            "reason": "Mốc thời gian model thực hiện prediction.",
        },
        {
            "column": "order_estimated_delivery_date",
            "role": "candidate_feature",
            "available_at_prediction": True,
            "reason": "Estimated deadline đã được xác định trước delivery.",
        },
        {
            "column": "order_delivered_carrier_date",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Xảy ra sau prediction point.",
        },
        {
            "column": "order_delivered_customer_date",
            "role": "target_construction_only",
            "available_at_prediction": False,
            "reason": "Actual outcome; chỉ dùng tạo target.",
        },
        {
            "column": "review_score",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Post-delivery information.",
        },
        {
            "column": "review_comment_title",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Post-delivery information.",
        },
        {
            "column": "review_comment_message",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Post-delivery information.",
        },
        {
            "column": "review_creation_date",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Post-delivery information.",
        },
        {
            "column": "review_answer_timestamp",
            "role": "forbidden_feature",
            "available_at_prediction": False,
            "reason": "Post-delivery information.",
        },
    ]
)

display(feature_availability_registry)

,column,role,available_at_prediction,reason
0,order_purchase_timestamp,candidate_feature,True,Có trước order approval.
1,order_approved_at,prediction_point,True,Mốc thời gian model thực hiện prediction.
2,order_estimated_delivery_date,candidate_feature,True,Estimated deadline đã được xác định trước delivery.
3,order_delivered_carrier_date,forbidden_feature,False,Xảy ra sau prediction point.
4,order_delivered_customer_date,target_construction_only,False,Actual outcome; chỉ dùng tạo target.
5,review_score,forbidden_feature,False,Post-delivery information.
6,review_comment_title,forbidden_feature,False,Post-delivery information.
7,review_comment_message,forbidden_feature,False,Post-delivery information.
8,review_creation_date,forbidden_feature,False,Post-delivery information.
9,review_answer_timestamp,forbidden_feature,False,Post-delivery information.


# 26. Tổng hợp join strategy sơ bộ

Đây chưa phải feature engineering. Mục tiêu chỉ là xác định cách join an toàn trong Phase 3.

In [35]:
join_strategy = pd.DataFrame(
    [
        {
            "table": "customers",
            "relationship_to_orders": "many-to-one via customer_id",
            "pre_aggregation_needed": False,
            "notes": "Join sau khi kiểm tra key integrity.",
        },
        {
            "table": "order_items",
            "relationship_to_orders": "one-to-many via order_id",
            "pre_aggregation_needed": True,
            "notes": "Aggregate item/order/product/seller information về order-level.",
        },
        {
            "table": "payments",
            "relationship_to_orders": "one-to-many via order_id",
            "pre_aggregation_needed": True,
            "notes": "Aggregate payment records về order-level trước khi merge.",
        },
        {
            "table": "reviews",
            "relationship_to_orders": "one-to-many or near one-to-one via order_id",
            "pre_aggregation_needed": True,
            "notes": "Không dùng làm model feature vì leakage; có thể dùng business analysis.",
        },
        {
            "table": "products",
            "relationship_to_orders": "through order_items",
            "pre_aggregation_needed": True,
            "notes": "Product attributes cần aggregate qua order_items.",
        },
        {
            "table": "sellers",
            "relationship_to_orders": "through order_items",
            "pre_aggregation_needed": True,
            "notes": "Một order có thể có nhiều seller.",
        },
        {
            "table": "geolocation",
            "relationship_to_orders": "through customer/seller zip prefix",
            "pre_aggregation_needed": True,
            "notes": "Aggregate raw geolocation thành one-row-per-zip lookup trước.",
        },
    ]
)

display(join_strategy)

,table,relationship_to_orders,pre_aggregation_needed,notes
0,customers,many-to-one via customer_id,False,Join sau khi kiểm tra key integrity.
1,order_items,one-to-many via order_id,True,Aggregate item/order/product/seller information về order-level.
2,payments,one-to-many via order_id,True,Aggregate payment records về order-level trước khi merge.
3,reviews,one-to-many or near one-to-one via order_id,True,Không dùng làm model feature vì leakage; có thể dùng business analysis.
4,products,through order_items,True,Product attributes cần aggregate qua order_items.
5,sellers,through order_items,True,Một order có thể có nhiều seller.
6,geolocation,through customer/seller zip prefix,True,Aggregate raw geolocation thành one-row-per-zip lookup trước.


# 27. Tạo summary artifact cho reports

Phần này chỉ export **metadata/quality summaries**, không sửa raw data.

In [36]:
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

dataset_overview.to_csv(
    REPORTS_METRICS_DIR / "dataset_overview.csv",
    index=False,
)

key_checks.to_csv(
    REPORTS_METRICS_DIR / "key_checks.csv",
    index=False,
)

fk_checks.to_csv(
    REPORTS_METRICS_DIR / "foreign_key_checks.csv",
    index=False,
)

timestamp_quality_summary.to_csv(
    REPORTS_METRICS_DIR / "timestamp_quality_checks.csv",
    index=False,
)

feature_availability_registry.to_csv(
    REPORTS_METRICS_DIR / "feature_availability_registry.csv",
    index=False,
)

print("✓ Đã export các summary CSV vào:", REPORTS_METRICS_DIR)

✓ Đã export các summary CSV vào: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/reports/metrics


# 28. Automated summary

Cell dưới đây in các kết luận định lượng chính từ dữ liệu vừa kiểm tra.

In [37]:
print("=" * 100)
print("DATASET UNDERSTANDING SUMMARY")
print("=" * 100)

print(f"\n1. Số bảng: {len(datasets)}")
print(f"2. Tổng số order rows: {len(orders):,}")
print(f"3. order_id unique: {orders['order_id'].is_unique}")
print(f"4. customer_id unique trong customers: {customers['customer_id'].is_unique}")
print(f"5. product_id unique: {products['product_id'].is_unique}")
print(f"6. seller_id unique: {sellers['seller_id'].is_unique}")

print("\n7. Orders có target timestamps đầy đủ:")
print(
    f"   {int(target_available_mask.sum()):,} / {len(orders_dt):,} "
    f"({target_available_mask.mean() * 100:.2f}%)"
)

print("\n8. Foreign-key issues:")
for _, row in fk_checks.iterrows():
    print(
        f"   {row['relationship']}: "
        f"{int(row['unknown_fk_rows']):,} unknown rows"
    )

print("\n9. Timestamp anomalies:")
for _, row in timestamp_quality_summary.iterrows():
    print(
        f"   {row['check']}: "
        f"{int(row['affected_rows']):,} rows"
    )

print("\n10. Join risks:")
print("   - order_items: one-to-many")
print("   - payments: one-to-many")
print("   - reviews: cần kiểm tra cardinality; không dùng làm model feature")
print("   - geolocation: zip prefix không nên giả định unique")

print("\n11. Leakage-critical columns:")
print(
    feature_availability_registry.loc[
        ~feature_availability_registry["available_at_prediction"],
        "column",
    ].tolist()
)

DATASET UNDERSTANDING SUMMARY

1. Số bảng: 9
2. Tổng số order rows: 99,441
3. order_id unique: True
4. customer_id unique trong customers: True
5. product_id unique: True
6. seller_id unique: True

7. Orders có target timestamps đầy đủ:
   96,476 / 99,441 (97.02%)

8. Foreign-key issues:
   orders.customer_id → customers.customer_id: 0 unknown rows
   order_items.order_id → orders.order_id: 0 unknown rows
   order_items.product_id → products.product_id: 0 unknown rows
   order_items.seller_id → sellers.seller_id: 0 unknown rows
   payments.order_id → orders.order_id: 0 unknown rows
   reviews.order_id → orders.order_id: 0 unknown rows

9. Timestamp anomalies:
   approval_before_purchase: 0 rows
   carrier_before_approval: 1,359 rows
   customer_delivery_before_carrier: 23 rows
   customer_delivery_before_purchase: 0 rows
   estimated_delivery_before_purchase: 0 rows

10. Join risks:
   - order_items: one-to-many
   - payments: one-to-many
   - reviews: cần kiểm tra cardinality; không d

# 29. Conclusions

Sau khi chạy notebook, cập nhật phần này dựa trên **kết quả thực tế**, không ghi theo giả định.

## 29.1 Dataset structure

- Dataset gồm nhiều relational tables.
- `orders` là bảng trung tâm cho bài toán delivery delay prediction.
- Final modeling grain dự kiến: **1 row = 1 order**.

## 29.2 Keys & relationships

Điền các phát hiện từ `key_checks` và `fk_checks`:

- `orders.order_id`: ...
- `customers.customer_id`: ...
- `products.product_id`: ...
- `sellers.seller_id`: ...
- `(order_items.order_id, order_item_id)`: ...
- Foreign-key integrity: ...

## 29.3 Important cardinality findings

Điền từ phần Cardinality:

- Items/order: ...
- Payments/order: ...
- Reviews/order: ...
- Geolocation rows/zip prefix: ...

## 29.4 Missing values & data quality

Ghi rõ:

- Missing nào là business-expected?
- Missing nào cần cleaning/imputation?
- Có timestamp anomaly nào cần loại/điều tra?
- Có exact duplicates không?
- Có invalid numeric values không?

## 29.5 Target feasibility

Target dự kiến:

`late_delivery = order_delivered_customer_date > order_estimated_delivery_date`

Cần ghi:

- Bao nhiêu orders có đủ target timestamps?
- Order statuses nào đủ/không đủ target?
- Population modeling cuối cùng sẽ được chốt ở bước Target Investigation / Cleaning.

## 29.6 Data leakage

Không được dùng làm model input:

- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- review-related fields
- bất kỳ dữ liệu nào chỉ xuất hiện sau `order_approved_at`

`order_delivered_customer_date` chỉ được dùng để tạo target.

## 29.7 Join risks

Không join trực tiếp tất cả raw tables.

Các bảng one-to-many phải aggregate riêng về order-level để tránh row multiplication:

- `order_items`
- `payments`
- seller/product information đi qua `order_items`
- raw `geolocation`

## 29.8 Modeling implications

- Prediction point: `order_approved_at`
- Modeling grain: order-level
- Validation nên ưu tiên chronological/time-based split
- Feature engineering phải tuân thủ feature availability tại prediction point
- Review data không được dùng làm model features

## 29.9 Open questions cho bước tiếp theo

1. Modeling population chính xác gồm những order statuses nào?
2. Có delivered order nào thiếu actual delivery timestamp không?
3. Cách xử lý timestamp anomalies?
4. Cách aggregate geolocation hợp lý nhất?
5. Các product attributes missing sẽ được xử lý thế nào?
6. Có cần historical seller/customer features không, và làm sao tránh temporal leakage?

# 30. Next step

Sau khi hoàn thành và đọc kết quả notebook này, bước tiếp theo là:

**Target Investigation + Data Quality / Cleaning Rules**

Chưa làm feature engineering và chưa train model.

Output mong muốn sau bước tiếp theo:

- xác định eligible modeling population;
- tạo target `late_delivery` đúng logic;
- định nghĩa cleaning rules có lý do;
- tạo dữ liệu sạch ở `data/interim/`;
- sau đó mới sang EDA và Feature Engineering.